In [1]:
import pandas as pd
from pandas.errors import PerformanceWarning
from sklearn.exceptions import UndefinedMetricWarning
import warnings
import sys
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

warnings.simplefilter('ignore', category=PerformanceWarning)
warnings.simplefilter('ignore', category=UndefinedMetricWarning)
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

This notebook trains or loads the already trained day-ahead tabular models.
Data is automatically loaded from the external APIs if not present.
Finally, each model is evaluated, including the best performing `XGBoost` model on the test set.

In [2]:
from src.models.train_day_ahead_tabular import train_day_ahead_tabular
from src.testing.data_split import DA_TARGET
from src.testing.datasets import load_day_ahead_tabular_split
from src.testing.evaluation import predict_tabular_model, evaluate
from src.testing.metrics import get_eval_metrics

eval_metrics = get_eval_metrics()
train, val, select, test = load_day_ahead_tabular_split()
model = train_day_ahead_tabular(train, val, "MAE", cache="MAE")

pred_train = predict_tabular_model(model, train, cache="TAB DA MAE Train")
pred_val = predict_tabular_model(model, val, cache="TAB DA MAE Val")
pred_select = predict_tabular_model(model, select, cache="TAB DA MAE Select")

y_train = pd.Series(train[DA_TARGET].values, index=train.index)
y_val = pd.Series(val[DA_TARGET].values, index=val.index)
y_select = pd.Series(select[DA_TARGET].values, index=select.index)
y_test = pd.Series(test[DA_TARGET].values, index=test.index)

metrics_train = evaluate(pred_train, y_train, eval_metrics)
metrics_val = evaluate(pred_val, y_val, eval_metrics)
metrics_select = evaluate(pred_select, y_select, eval_metrics)

display(metrics_train, metrics_val, metrics_select)

pred_test = predict_tabular_model(model, test, cache="TAB DA MAE Test", model_names=["XGBoost"])
metrics_test = evaluate(pred_test, y_test, eval_metrics)
display(metrics_test)

/Users/pavelholman/FIT/bp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-16 18:45:53,515 - INFO - Loading day ahead data
2025-05-16 18:45:53,515 - INFO - Loading data from /Users/pavelholman/FIT/bp/data/interim/data.csv
2025-05-16 18:45:55,027 - INFO - Data split info:
Original DataFrame size: 6408
Train size: 4248 (66.29%)
Validation size: 744 (11.61%)
Select size: 672 (10.49%)
Test size: 744 (11.61%)
2025-05-16 18:45:55,029 - INFO - Loading model from /Users/pavelholman/FIT/bp/models/day_ahead/tabular/MAE
/Users/pavelholman/FIT/bp/.venv/lib/python3.12/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be 

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val          eval_metric  pred_time_val    fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  WeightedEnsemble_L2 -22.087777  mean_absolute_error       0.152952  376.199044                0.000240           0.013018            2       True          5
1             CatBoost -22.207982  mean_absolute_error       0.051033   41.800351                0.051033          41.800351            1       True          3
2         RandomForest -25.797168  mean_absolute_error       0.053354  293.810218                0.053354         293.810218            1       True          2
3              XGBoost -26.141568  mean_absolute_error       0.048325   40.575457                0.048325          40.575457            1       True          4
4           KNeighbors -38.519753  mean_absolute_error       0.248092    0.352867                0.248092           0.352867            1 

,mean_absolute_error,mean_absolute_percentage_error,root_mean_squared_error,directional accuracy,peak detection accuracy,peak detection precision,peak detection recall,peak detection f1
KNeighbors,-10.094417,-0.211101,-15.616007,0.614925,0.928571,0.857143,0.352941,0.500000
RandomForest,-4.738890,-0.100539,-6.708485,0.874627,0.970238,0.928571,0.764706,0.838710
CatBoost,-14.575122,-0.327263,-19.651234,0.719403,0.916667,0.593750,0.558824,0.575758
XGBoost,-14.745692,-0.317431,-20.301965,0.656716,0.919643,0.769231,0.294118,0.425532
WeightedEnsemble_L2,-13.992259,-0.315071,-18.958438,0.731343,0.922619,0.633333,0.558824,0.593750


,mean_absolute_error,mean_absolute_percentage_error,root_mean_squared_error,directional accuracy,peak detection accuracy,peak detection precision,peak detection recall,peak detection f1
KNeighbors,-42.030375,-0.288243,-62.549276,0.552239,0.886905,0.166667,0.029412,0.050000
RandomForest,-27.115465,-0.182622,-45.564105,0.602985,0.886905,0.409091,0.264706,0.321429
CatBoost,-22.845665,-0.154649,-39.770674,0.740299,0.877976,0.400000,0.411765,0.405797
XGBoost,-28.936424,-0.189592,-47.357408,0.704478,0.883929,0.333333,0.147059,0.204082
WeightedEnsemble_L2,-23.030081,-0.154264,-40.251632,0.752239,0.883929,0.413793,0.352941,0.380952


,mean_absolute_error,mean_absolute_percentage_error,root_mean_squared_error,directional accuracy,peak detection accuracy,peak detection precision,peak detection recall,peak detection f1
KNeighbors,-38.882095,-0.320076,-47.873531,0.588060,0.892857,0.250000,0.029412,0.052632
RandomForest,-28.616289,-0.251700,-37.124155,0.653731,0.797619,0.250000,0.500000,0.333333
CatBoost,-25.937019,-0.227802,-36.692331,0.683582,0.812500,0.316456,0.735294,0.442478
XGBoost,-22.004068,-0.188682,-29.304980,0.668657,0.880952,0.392857,0.323529,0.354839
WeightedEnsemble_L2,-24.432546,-0.215016,-34.076584,0.671642,0.833333,0.338235,0.676471,0.450980


2025-05-16 18:45:55,876 - INFO - Loading cached predictions for XGBoost


,mean_absolute_error,mean_absolute_percentage_error,root_mean_squared_error,directional accuracy,peak detection accuracy,peak detection precision,peak detection recall,peak detection f1
XGBoost,-24.004572,-10.888764,-32.961286,0.692537,0.916667,0.75,0.264706,0.391304
